In [ ]:
# ============================================
# MLflow Starter - Autolog, Manual Log, Store & Load
# Dataset: Breast Cancer (sklearn built-in)
# ============================================

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

# --- Load Dataset ---
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
print(f"Dataset shape: {df.shape}")
print(f"Target classes: {dict(enumerate(data.target_names))}")
print(df.head())

X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# --- PART 1: Autolog ---
print("=" * 50)
print("PART 1: MLflow Autolog")
print("=" * 50)

mlflow.autolog()

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
predictions = rf.predict(X_test)
print(f"Autolog Accuracy: {np.mean(predictions == y_test):.4f}")

# --- PART 2: Manual Logging ---
print("\n" + "=" * 50)
print("PART 2: Manual Logging")
print("=" * 50)

mlflow.autolog(disable=True)

with mlflow.start_run(run_name="manual_rf_breast_cancer") as run:
    rf2 = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    rf2.fit(X_train, y_train)
    preds = rf2.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    # Log parameters manually
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 8)

    # Log metrics manually
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # Log model with signature
    signature = infer_signature(X_train, rf2.predict(X_train))
    mlflow.sklearn.log_model(rf2, "model", signature=signature)

    print(f"\nRun ID: {run.info.run_id}")

# --- PART 3: Load the saved model ---
print("\n" + "=" * 50)
print("PART 3: Load Saved Model")
print("=" * 50)

model_uri = f"runs:/{run.info.run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)
loaded_preds = loaded_model.predict(X_test)
print(f"Loaded Model Accuracy: {accuracy_score(y_test, loaded_preds):.4f}")
print("Model loaded and verified successfully!")

Dataset shape: (0, 1)
Empty DataFrame
Columns: [404: Not Found]
Index: []


KeyError: "['target'] not found in axis"